# ops

> Column operations for MXFrame - arithmetic, comparisons, aggregations.
> Uses MAX Engine for compute (CPU and GPU), PyArrow for storage.

In [27]:
#| default_exp ops

In [28]:
%pip install -e .. -q

Note: you may need to restart the kernel to use updated packages.


In [29]:
#| export
import pyarrow as pa
import pyarrow.compute as pc  # Only for filter/take operations
import numpy as np
from typing import Union, List, Optional, Any, Literal
from mxframe.core import MXFrame
from mxframe.max_ops import (
    DeviceType,
    max_sum, max_mean, max_min, max_max, max_count, max_std,
    max_add, max_sub, max_mul, max_div, max_neg,
    max_greater, max_greater_eq, max_less, max_less_eq, max_equal, max_not_equal,
    max_and, max_or, max_not
)

In [30]:
#| export
Scalar = Union[int, float, bool]

In [31]:
#| export
class MXColumn:
    """
    Wrapper around PyArrow ChunkedArray with arithmetic and comparison operators.
    
    Uses MAX Engine for ALL compute (CPU or GPU), PyArrow only for storage.
    
    Example:
        >>> col = MXColumn(pa.chunked_array([[1, 2, 3]]))
        >>> result = col * 2 + 1
        >>> mask = col > 1
        >>> total = col.sum(device="gpu")  # Use GPU for aggregation
    """
    
    def __init__(self, data: Union[pa.ChunkedArray, pa.Array, list, np.ndarray], 
                 name: str = "", device: DeviceType = "cpu"):
        if isinstance(data, pa.ChunkedArray):
            self._data = data
        elif isinstance(data, pa.Array):
            self._data = pa.chunked_array([data])
        elif isinstance(data, (list, np.ndarray)):
            self._data = pa.chunked_array([pa.array(data)])
        else:
            raise TypeError(f"Cannot create MXColumn from {type(data)}")
        self._name = name
        self._device = device
    
    @property
    def name(self) -> str:
        return self._name
    
    @property
    def dtype(self) -> pa.DataType:
        return self._data.type
    
    @property
    def device(self) -> DeviceType:
        return self._device
    
    def __len__(self) -> int:
        return len(self._data)
    
    def to_arrow(self) -> pa.ChunkedArray:
        return self._data
    
    def to_numpy(self) -> np.ndarray:
        return self._data.to_numpy(zero_copy_only=False)
    
    def to_pylist(self) -> list:
        return self._data.to_pylist()
    
    def to(self, device: DeviceType) -> 'MXColumn':
        """Return column targeting a different device."""
        return MXColumn(self._data, self._name, device)
    
    def _other_numpy(self, other: Union['MXColumn', Scalar]) -> np.ndarray:
        """Convert other operand to numpy array."""
        if isinstance(other, MXColumn):
            return other.to_numpy()
        else:
            return other  # Scalar - MAX functions handle it
    
    # ==================== Arithmetic (MAX Engine) ====================
    
    def __add__(self, other: Union['MXColumn', Scalar]) -> 'MXColumn':
        result = max_add(self.to_numpy(), self._other_numpy(other), self._device)
        return MXColumn(pa.array(result), device=self._device)
    
    def __radd__(self, other: Scalar) -> 'MXColumn':
        result = max_add(self.to_numpy(), other, self._device)
        return MXColumn(pa.array(result), device=self._device)
    
    def __sub__(self, other: Union['MXColumn', Scalar]) -> 'MXColumn':
        result = max_sub(self.to_numpy(), self._other_numpy(other), self._device)
        return MXColumn(pa.array(result), device=self._device)
    
    def __rsub__(self, other: Scalar) -> 'MXColumn':
        # other - self = -(self - other)
        result = max_sub(np.full(len(self), other, dtype=np.float32), self.to_numpy(), self._device)
        return MXColumn(pa.array(result), device=self._device)
    
    def __mul__(self, other: Union['MXColumn', Scalar]) -> 'MXColumn':
        result = max_mul(self.to_numpy(), self._other_numpy(other), self._device)
        return MXColumn(pa.array(result), device=self._device)
    
    def __rmul__(self, other: Scalar) -> 'MXColumn':
        result = max_mul(self.to_numpy(), other, self._device)
        return MXColumn(pa.array(result), device=self._device)
    
    def __truediv__(self, other: Union['MXColumn', Scalar]) -> 'MXColumn':
        result = max_div(self.to_numpy(), self._other_numpy(other), self._device)
        return MXColumn(pa.array(result), device=self._device)
    
    def __rtruediv__(self, other: Scalar) -> 'MXColumn':
        result = max_div(np.full(len(self), other, dtype=np.float32), self.to_numpy(), self._device)
        return MXColumn(pa.array(result), device=self._device)
    
    def __neg__(self) -> 'MXColumn':
        result = max_neg(self.to_numpy(), self._device)
        return MXColumn(pa.array(result), device=self._device)
    
    # ==================== Comparisons (MAX Engine) ====================
    
    def __gt__(self, other: Union['MXColumn', Scalar]) -> 'MXColumn':
        result = max_greater(self.to_numpy(), self._other_numpy(other), self._device)
        return MXColumn(pa.array(result), device=self._device)
    
    def __ge__(self, other: Union['MXColumn', Scalar]) -> 'MXColumn':
        result = max_greater_eq(self.to_numpy(), self._other_numpy(other), self._device)
        return MXColumn(pa.array(result), device=self._device)
    
    def __lt__(self, other: Union['MXColumn', Scalar]) -> 'MXColumn':
        result = max_less(self.to_numpy(), self._other_numpy(other), self._device)
        return MXColumn(pa.array(result), device=self._device)
    
    def __le__(self, other: Union['MXColumn', Scalar]) -> 'MXColumn':
        result = max_less_eq(self.to_numpy(), self._other_numpy(other), self._device)
        return MXColumn(pa.array(result), device=self._device)
    
    def __eq__(self, other: Union['MXColumn', Scalar]) -> 'MXColumn':
        result = max_equal(self.to_numpy(), self._other_numpy(other), self._device)
        return MXColumn(pa.array(result), device=self._device)
    
    def __ne__(self, other: Union['MXColumn', Scalar]) -> 'MXColumn':
        result = max_not_equal(self.to_numpy(), self._other_numpy(other), self._device)
        return MXColumn(pa.array(result), device=self._device)
    
    # ==================== Boolean Ops (MAX Engine) ====================
    
    def __and__(self, other: 'MXColumn') -> 'MXColumn':
        result = max_and(self.to_numpy(), other.to_numpy(), self._device)
        return MXColumn(pa.array(result), device=self._device)
    
    def __or__(self, other: 'MXColumn') -> 'MXColumn':
        result = max_or(self.to_numpy(), other.to_numpy(), self._device)
        return MXColumn(pa.array(result), device=self._device)
    
    def __invert__(self) -> 'MXColumn':
        result = max_not(self.to_numpy(), self._device)
        return MXColumn(pa.array(result), device=self._device)
    
    # ==================== Aggregations (MAX Engine) ====================
    
    def sum(self, device: DeviceType = None) -> Scalar:
        """Sum all values using MAX Engine."""
        dev = device or self._device
        return max_sum(self.to_numpy(), device=dev)
    
    def mean(self, device: DeviceType = None) -> float:
        """Mean of all values using MAX Engine."""
        dev = device or self._device
        return max_mean(self.to_numpy(), device=dev)
    
    def min(self, device: DeviceType = None) -> Scalar:
        """Minimum value using MAX Engine."""
        dev = device or self._device
        return max_min(self.to_numpy(), device=dev)
    
    def max(self, device: DeviceType = None) -> Scalar:
        """Maximum value using MAX Engine."""
        dev = device or self._device
        return max_max(self.to_numpy(), device=dev)
    
    def count(self, device: DeviceType = None) -> int:
        """Count elements using MAX Engine."""
        dev = device or self._device
        return max_count(self.to_numpy(), device=dev)
    
    def std(self, device: DeviceType = None) -> float:
        """Standard deviation using MAX Engine."""
        dev = device or self._device
        return max_std(self.to_numpy(), device=dev)
    
    # ==================== Display ====================
    
    def __repr__(self) -> str:
        name_str = f"'{self._name}'" if self._name else ""
        return f"MXColumn{name_str}({len(self)}, {self.dtype}, device={self._device})"
    
    def __str__(self) -> str:
        return str(self._data)

In [32]:
# Test MXColumn arithmetic
col = MXColumn([1, 2, 3, 4, 5])
print(f"col = {col.to_pylist()}")
print(f"col + 10 = {(col + 10).to_pylist()}")
print(f"col * 2 = {(col * 2).to_pylist()}")
print(f"col * 2 + 1 = {(col * 2 + 1).to_pylist()}")
print(f"1 - col = {(1 - col).to_pylist()}")

col = [1, 2, 3, 4, 5]


col + 10 = [11.0, 12.0, 13.0, 14.0, 15.0]
col * 2 = [2.0, 4.0, 6.0, 8.0, 10.0]
col * 2 + 1 = [3.0, 5.0, 7.0, 9.0, 11.0]
1 - col = [0.0, -1.0, -2.0, -3.0, -4.0]


In [33]:
# Test MXColumn comparisons
col = MXColumn([1, 2, 3, 4, 5])
print(f"col > 2 = {(col > 2).to_pylist()}")
print(f"col <= 3 = {(col <= 3).to_pylist()}")
print(f"(col > 1) & (col < 5) = {((col > 1) & (col < 5)).to_pylist()}")

col > 2 = [False, False, True, True, True]
col <= 3 = [True, True, True, False, False]
(col > 1) & (col < 5) = [False, True, True, True, False]


In [34]:
# Test MXColumn aggregations with MAX Engine
col = MXColumn([1, 2, 3, 4, 5])
print(f"sum (auto) = {col.sum()}")
print(f"sum (cpu) = {col.sum(device='cpu')}")
print(f"sum (gpu) = {col.sum(device='gpu')}")
print(f"mean = {col.mean()}")
print(f"min = {col.min()}, max = {col.max()}")
print(f"count = {col.count()}")

sum (auto) = 15.0
sum (cpu) = 15.0
sum (gpu) = 15.0
mean = 3.0
min = 1.0, max = 5.0
count = 5


In [35]:
#| export
def filter_frame(frame: MXFrame, mask: Union[MXColumn, pa.ChunkedArray]) -> MXFrame:
    """
    Filter MXFrame rows by boolean mask.
    
    Args:
        frame: MXFrame to filter
        mask: Boolean mask (MXColumn or ChunkedArray)
        
    Returns:
        New MXFrame with filtered rows
    """
    mask_data = mask._data if isinstance(mask, MXColumn) else mask
    filtered_table = pc.filter(frame.to_arrow(), mask_data)
    return MXFrame(filtered_table)

In [36]:
#| export
def assign_columns(frame: MXFrame, **kwargs) -> MXFrame:
    """
    Create new MXFrame with additional/replaced columns.
    
    Args:
        frame: Source MXFrame
        **kwargs: Column name -> MXColumn or array-like
        
    Returns:
        New MXFrame with added columns
        
    Example:
        >>> new_frame = assign_columns(frame, 
        ...     disc_price=frame['price'] * (1 - frame['disc']))
    """
    table = frame.to_arrow()
    
    for name, col in kwargs.items():
        if isinstance(col, MXColumn):
            arr = col.to_arrow()
        elif isinstance(col, pa.ChunkedArray):
            arr = col
        elif isinstance(col, (list, np.ndarray)):
            arr = pa.chunked_array([pa.array(col)])
        else:
            raise TypeError(f"Cannot assign column from {type(col)}")
        
        # Check if column exists, replace or append
        if name in table.column_names:
            idx = table.column_names.index(name)
            table = table.set_column(idx, name, arr)
        else:
            table = table.append_column(name, arr)
    
    return MXFrame(table)

In [37]:
#| export
def col(frame: MXFrame, name: str) -> MXColumn:
    """Get column from MXFrame as MXColumn."""
    return MXColumn(frame.to_arrow().column(name), name=name)

In [38]:
# Test filter and assign with TPC-H Q1 pattern
frame = MXFrame({
    'qty': [1, 2, 3, 4, 5],
    'price': [10.0, 20.0, 30.0, 40.0, 50.0],
    'disc': [0.1, 0.05, 0.0, 0.15, 0.2],
    'shipdate': [10000, 10100, 10200, 10300, 10400],  # epoch days
})

print("Original frame:")
print(frame)
print()

# Filter: shipdate <= 10200
mask = col(frame, 'shipdate') <= 10200
filtered = filter_frame(frame, mask)
print(f"Filtered (shipdate <= 10200): {filtered.num_rows} rows")
print()

# Assign: disc_price = price * (1 - disc)
price = col(frame, 'price')
disc = col(frame, 'disc')
with_disc_price = assign_columns(frame, disc_price=price * (1 - disc))
print(f"With disc_price column: {with_disc_price.column_names}")
print(f"disc_price = {col(with_disc_price, 'disc_price').to_pylist()}")

Original frame:
MXFrame(5 rows, 4 cols) [qty: int64?, price: float64?, disc: float64?, shipdate: int64?]

Filtered (shipdate <= 10200): 3 rows

With disc_price column: ['qty', 'price', 'disc', 'shipdate', 'disc_price']
disc_price = [9.0, 19.0, 30.0, 34.0, 40.0]


In [39]:
#| hide
from nbdev import nbdev_export
nbdev_export()